# Only for windows

In [1]:

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


In [2]:
import torch
import torchvision.transforms as T
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import requests
from io import BytesIO


In [10]:
from huggingface_hub import hf_hub_download

ckpt = hf_hub_download(repo_id="huggan/pix2pix-maps", filename="pytorch_model.bin")

c:\Users\Stefano\anaconda3\envs\GenAi\Lib\site-packages\huggingface_hub\file_download.py:121: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Stefano\.cache\huggingface\hub\models--huggan--pix2pix-maps. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [ ]:
import torch
import torch.nn as nn


class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, down=True, activation="relu", use_dropout=False):
        super().__init__()
        self.down = down

        if down:
            self.conv = nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)
        else:
            self.conv = nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False)

        self.bn = nn.BatchNorm2d(out_ch)

        if activation == "relu":
            self.act = nn.ReLU(True)
        else:
            self.act = nn.LeakyReLU(0.2, True)

        self.use_dropout = use_dropout
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.act(self.bn(self.conv(x)))
        if self.use_dropout:
            x = self.dropout(x)
        return x


class UNetGenerator(nn.Module):
    """UNet 256 from Pix2Pix original paper."""

    def __init__(self, input_nc=3, output_nc=3, ngf=64):
        super().__init__()

        # Encoder
        self.down1 = nn.Conv2d(input_nc, ngf, 4, 2, 1)  # no BN
        self.down2 = UNetBlock(ngf, ngf * 2, down=True, activation="lrelu")
        self.down3 = UNetBlock(ngf * 2, ngf * 4, down=True, activation="lrelu")
        self.down4 = UNetBlock(ngf * 4, ngf * 8, down=True, activation="lrelu")
        self.down5 = UNetBlock(ngf * 8, ngf * 8, down=True, activation="lrelu")
        self.down6 = UNetBlock(ngf * 8, ngf * 8, down=True, activation="lrelu", use_dropout=True)
        self.down7 = UNetBlock(ngf * 8, ngf * 8, down=True, activation="lrelu", use_dropout=True)
        self.down8 = UNetBlock(ngf * 8, ngf * 8, down=True, activation="lrelu", use_dropout=True)

        # Decoder
        self.up1 = UNetBlock(ngf * 8, ngf * 8, down=False, use_dropout=True)
        self.up2 = UNetBlock(ngf * 16, ngf * 8, down=False, use_dropout=True)
        self.up3 = UNetBlock(ngf * 16, ngf * 8, down=False)
        self.up4 = UNetBlock(ngf * 16, ngf * 8, down=False)
        self.up5 = UNetBlock(ngf * 16, ngf * 4, down=False)
        self.up6 = UNetBlock(ngf * 8, ngf * 2, down=False)
        self.up7 = UNetBlock(ngf * 4, ngf, down=False)

        self.up8 = nn.ConvTranspose2d(ngf * 2, output_nc, 4, 2, 1)
        self.tanh = nn.Tanh()

    def forward(self, x):
        # Encoder
        e1 = self.down1(x)
        e2 = self.down2(e1)
        e3 = self.down3(e2)
        e4 = self.down4(e3)
        e5 = self.down5(e4)
        e6 = self.down6(e5)
        e7 = self.down7(e6)
        e8 = self.down8(e7)

        # Decoder + skip connections
        d1 = self.up1(e8)
        d1 = torch.cat([d1, e7], 1)

        d2 = self.up2(d1)
        d2 = torch.cat([d2, e6], 1)

        d3 = self.up3(d2)
        d3 = torch.cat([d3, e5], 1)

        d4 = self.up4(d3)
        d4 = torch.cat([d4, e4], 1)

        d5 = self.up5(d4)
        d5 = torch.cat([d5, e3], 1)

        d6 = self.up6(d5)
        d6 = torch.cat([d6, e2], 1)

        d7 = self.up7(d6)
        d7 = torch.cat([d7, e1], 1)

        out = self.up8(d7)
        return self.tanh(out)


In [13]:
model = Pix2PixUNet()
state_dict = torch.load(ckpt, map_location="cpu")
model.load_state_dict(state_dict)
model.eval()

c:\Users\Stefano\anaconda3\envs\GenAi\Lib\site-packages\torch\hub.py:335: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(


Downloading: "https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix/zipball/master" to C:\Users\Stefano/.cache\torch\hub\master.zip


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Stefano/.cache\\torch\\hub\\junyanz_pytorch-CycleGAN-and-pix2pix_master\\hubconf.py'

In [4]:
transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor()
])

def preprocess_sketch(path):
    img = Image.open(path).convert("RGB")
    img = transform(img)
    img = img.unsqueeze(0)
    return img


In [6]:
def generate_pix2pix(sketch_path):
    sketch_tensor = preprocess_sketch(sketch_path)
    with torch.no_grad():
        output = model(sketch_tensor)[0] * 0.5 + 0.5
        
    output_img = T.ToPILImage()(output.cpu())
    return output_img


In [8]:
sketch_path = "sketch.jpg"  # inserire file
output = generate_pix2pix(sketch_path)

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.title("Sketch Input")
plt.imshow(Image.open(sketch_path))
plt.axis("off")

plt.subplot(1,2,2)
plt.title("Pix2Pix Output")
plt.imshow(output)
plt.axis("off")

plt.show()


NameError: name 'model' is not defined